<div dir="rtl">

# ‏02_benchmark‏: قياس السرعة (‏Inference Time‏ و‏FPS‏)

**الهدف:** نعرف قديش بتاخد كل مرحلة من مراحل النظام وقت، وقديش إطار بيعالج النظام كامل بالثانية.

**ليش منقيس على فيديو مسجّل مو على الكاميرا؟** لأنه الفيديو نفسه كل مرة. إذا عدّلنا شي وقسنا من جديد، الفرق سببه التعديل مو إنه المشهد تغيّر (نفس فكرة الـ ‏Seed‏).

**المراحل اللي منقيسها لكل إطار:**
1. ‏face_detect‏: ‏SCRFD‏ بيلاقي الوجوه.
2. ‏face_embed‏: ‏ArcFace‏ بيطلّع بصمة ‏512‏ رقم لكل وجه.
3. ‏classify‏: النموذج ‏B‏ بيقرر علاء ولا لأ.
4. ‏segment‏: ‏YOLO-seg‏ بيطلّع قناع لكل جسم.
5. ‏link‏: كل جسم بياخد لون الوجه اللي جوّاه.
6. ‏draw‏: الرسم متل الكاميرا الحية.

**الرسوم:** رسمين بس، لأنه كل واحد بيجاوب سؤال ما بيجاوب عليه الجدول:
- أعمدة لزمن كل مرحلة: **وين عنق الزجاجة؟**
- خط للزمن الكلي لكل إطار: **هل السرعة ثابتة ولا في قفزات؟**

**شغّل الـ ‏Notebook‏ من مجلد ‏face_recognition‏**، وبنفس الـ ‏Kernel‏ تبع الـ ‏venv‏.

</div>

<div dir="rtl">

## الخلية ‏1‏: الإعدادات وتحميل النماذج

- منجيب **نفس النماذج** اللي بتستعملها الكاميرا الحية.
- منفصل ‏InsightFace‏ لجزئين: ‏det_model‏ (كشف الوجه) و‏rec_model‏ (البصمة)، لنقيس كل واحد لحاله.
- الدوال الجاهزة (‏person_masks‏، ‏body_colors‏، ‏draw‏) **منستوردها** من الملفات الأصلية بدل ما ننسخها، فمنقيس نفس الكود اللي بيشتغل لايف.
- ‏WARMUP‏: أول إطارات بتكون أبطأ لأنه الكرت عم يجهّز حاله، فمنشغّلهم بدون ما نسجّلهم.

</div>

In [ ]:
import json
import time
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from insightface.app import FaceAnalysis
from insightface.app.common import Face
from ultralytics import YOLO

import config as C
from live_camera import load_scorer, draw, GREEN, RED
from live_camera_seg import SEG_MODEL, DEVICE, ALPHA, person_masks, body_colors

VIDEO = Path("data/raw/alaa_test/alaa_test.mp4")
FRAME_SIZE = (1280, 720)      # same size as the live camera
MAX_FRAMES = 300              # frames that are measured
WARMUP = 10                   # first frames are slower (GPU setup): run them, do not measure them
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

scorer, model_name, threshold = load_scorer()
app = FaceAnalysis(
    name=C.MODEL_NAME,
    allowed_modules=["detection", "recognition"],
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=C.DET_SIZE)
det_model = app.det_model                    # SCRFD: finds faces + 5 landmarks
rec_model = app.models["recognition"]        # ArcFace: face -> 512-number embedding
seg_model = YOLO(SEG_MODEL)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("InsightFace:", det_model.session.get_providers()[0], "| YOLO device:", DEVICE)
print(f"model={model_name} threshold={threshold}")

<div dir="rtl">

## الخلية ‏2‏: تحميل الإطارات بالذاكرة

- منقرأ ‏310‏ إطار (‏10‏ تحمية + ‏300‏ قياس) ومنصغّرهم لـ ‏1280×720‏، نفس حجم الكاميرا الحية.
- **ليش بالذاكرة أول؟** لأنه ما بدنا نقيس وقت قراءة الملف من الديسك، بدنا نقيس النماذج بس.
- ‏assert‏ حارس: إذا الفيديو ما انفتح أو قصير، بيوقف فوراً بدل ما يكمل بنتائج غلط.

**✅ تحقق:** لازم يطبع ‏loaded 310 frames of size 1280x720‏.

</div>

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), f"Cannot open {VIDEO}"
frames = []
while len(frames) < WARMUP + MAX_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    frames.append(cv2.resize(frame, FRAME_SIZE))
cap.release()

assert len(frames) > WARMUP, "video too short"
print(f"loaded {len(frames)} frames of size {frames[0].shape[1]}x{frames[0].shape[0]}")

<div dir="rtl">

## الخلية ‏3‏: القياس

- ‏run_frame‏ بتمرّق الإطار على المراحل الست، وبعد كل مرحلة بتسجّل الوقت بـ ‏time.perf_counter()‏ (ساعة دقيقة جداً).
- ‏np.diff‏ بيطرح كل وقت من اللي بعده، فبيطلع زمن كل مرحلة. ومنضرب بـ ‏1000‏ لنحوّل لميلي ثانية.
- منشغّل أول ‏10‏ إطارات (التحمية) بدون تسجيل، وبعدين منسجّل الـ ‏300‏.
- كمان منعدّ لكل إطار: كم وجه، كم جسم، وكم وجه انقبل كعلاء. هدول للتحقق بالخلية ‏4‏.

**ملاحظة:** هون ما في تتبع ولا ‏Smoothing‏ متل الكاميرا الحية، لأنه زمنهم صغير جداً (عمليات حسابية بسيطة)، والهدف قياس النماذج.

</div>

In [ ]:
STAGES = ["face_detect", "face_embed", "classify", "segment", "link", "draw"]

def run_frame(frame):
    """One frame through the full live pipeline, with a stopwatch between stages."""
    t = [time.perf_counter()]

    # 1) SCRFD: face boxes + landmarks
    bboxes, kpss = det_model.detect(frame, max_num=0, metric="default")
    t.append(time.perf_counter())

    # 2) ArcFace: one embedding per good face
    faces = []
    for i in range(bboxes.shape[0]):
        if bboxes[i, 4] < C.MIN_DET_SCORE:
            continue
        face = Face(bbox=bboxes[i, :4], kps=kpss[i] if kpss is not None else None, det_score=bboxes[i, 4])
        rec_model.get(frame, face)
        faces.append(face)
    t.append(time.perf_counter())

    # 3) model B: Alaa or not (biggest face first, like live_camera.py)
    face_results = []
    for f in sorted(faces, key=lambda f: -(f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1])):
        s = scorer(f.normed_embedding)
        face_results.append((f.bbox, "", GREEN if s >= threshold else RED, s))
    t.append(time.perf_counter())

    # 4) YOLO-seg: one mask per person
    masks = person_masks(seg_model, frame)
    t.append(time.perf_counter())

    # 5) give each body the color of its face
    colors = body_colors(face_results, masks)
    t.append(time.perf_counter())

    # 6) draw, like the live app
    overlay = frame.copy()
    for m, c in zip(masks, colors):
        overlay[m.astype(bool)] = c
    view = cv2.flip(cv2.addWeighted(overlay, ALPHA, frame, 1 - ALPHA, 0), 1)
    draw(view, face_results, True, mirror_width=frame.shape[1])
    t.append(time.perf_counter())

    stage_ms = np.diff(t) * 1000
    n_alaa = sum(1 for r in face_results if r[3] >= threshold)
    return stage_ms, len(faces), len(masks), n_alaa

for f in frames[:WARMUP]:                    # warm-up: run, do not record
    run_frame(f)

times, n_faces, n_people, n_alaa = [], [], [], []
for f in frames[WARMUP:]:
    ms, nf, npp, na = run_frame(f)
    times.append(ms); n_faces.append(nf); n_people.append(npp); n_alaa.append(na)

times = np.array(times)          # shape: (frames, stages)
total = times.sum(axis=1)        # total ms per frame
print(f"measured {len(times)} frames")

<div dir="rtl">

## الخلية ‏4‏: الجدول والتحقق

**أعمدة الجدول:**
- ‏mean‏: المتوسط.
- ‏p50‏: الوسيط، نص الإطارات أسرع منه ونصها أبطأ.
- ‏p95‏: ‏95%‏ من الإطارات أسرع منه. **هاد المهم للايف**، لأنه القفزات بتبيّن كتقطيع حتى لو المتوسط حلو.
- ‏share‏: نسبة المرحلة من الوقت الكلي. أكبر نسبة = عنق الزجاجة.

**‏FPS = 1000 ÷ الزمن الكلي بالميلي ثانية.** وهاد للنماذج بس، بدون الكاميرا. الكاميرا نفسها غالباً محدودة بتقريباً ‏30‏، فإذا الرقم هون أعلى، عنق الزجاجة باللايف بيصير الكاميرا.

**✅ تحقق:** الفيديو كله لعلاء، فلازم معظم الإطارات فيها وجه انقبل كعلاء. إذا طلعت النسبة واطية، في غلط بالقياس نفسه، فما منثق بالأرقام.

بالآخر منحفظ كل شي بـ ‏results/benchmark.json‏ لنستعمله بالـ ‏README‏.

</div>

In [ ]:
print(f"{'stage':<12}{'mean ms':>9}{'p50':>8}{'p95':>8}{'share':>8}")
for i, s in enumerate(STAGES):
    col = times[:, i]
    print(f"{s:<12}{col.mean():>9.2f}{np.percentile(col, 50):>8.2f}{np.percentile(col, 95):>8.2f}"
          f"{col.mean() / total.mean():>8.0%}")
p50, p95 = np.percentile(total, 50), np.percentile(total, 95)
print(f"{'TOTAL':<12}{total.mean():>9.2f}{p50:>8.2f}{p95:>8.2f}")

fps_mean, fps_p95 = 1000 / total.mean(), 1000 / p95
print(f"\nFPS (pipeline only, no camera): mean ~{fps_mean:.0f}, slowest 5% of frames ~{fps_p95:.0f}")

# sanity check: alaa_test is a video of Alaa, so most frames should have one face accepted as Alaa
n_faces, n_people, n_alaa = map(np.array, (n_faces, n_people, n_alaa))
print(f"frames with a face: {(n_faces > 0).mean():.0%} | Alaa accepted: {(n_alaa > 0).mean():.0%} "
      f"| avg people per frame: {n_people.mean():.2f}")

summary = {
    "frames": int(len(times)), "frame_size": FRAME_SIZE, "warmup": WARMUP,
    "device_yolo": str(DEVICE), "model": model_name, "threshold": threshold,
    "stages_ms": {s: {"mean": float(times[:, i].mean()), "p95": float(np.percentile(times[:, i], 95))}
                  for i, s in enumerate(STAGES)},
    "total_ms": {"mean": float(total.mean()), "p50": float(p50), "p95": float(p95)},
    "fps": {"mean": float(fps_mean), "p95_frames": float(fps_p95)},
}
(RESULTS / "benchmark.json").write_text(json.dumps(summary, indent=2))
print("saved results/benchmark.json")

<div dir="rtl">

## الخلية ‏5‏: الرسمين

**الرسم ‏1‏ (أعمدة):** طول العمود = متوسط زمن المرحلة، والشحطة السودا = ‏p95‏. العمود الأطول هو عنق الزجاجة، وهو أول مكان منحسّنه إذا بدنا سرعة أكتر.

**الرسم ‏2‏ (خط):** الزمن الكلي لكل إطار ورا بعض. الخط المتقطع = المتوسط، والخط المنقّط = ‏p95‏. إذا الخط مستوي، السرعة ثابتة. وإذا فيه قمم، في إطارات بتتأخر (تقطيع).

**ما رسمنا:** ‏Precision‏ و‏Recall‏ (كلهم ‏1.0‏، الرسم ما بيضيف)، و‏FPS‏ لحاله (بينقرأ من الرسم ‏2‏).

بينحفظ الرسم بـ ‏results/benchmark.png‏.

</div>

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Chart 1: where does the time go?
means = times.mean(axis=0)
p95s = np.percentile(times, 95, axis=0)
ax1.bar(STAGES, means, label="mean")
ax1.scatter(STAGES, p95s, marker="_", s=700, color="black", label="p95")
ax1.set_ylabel("ms per frame")
ax1.set_title("Inference time per stage")
ax1.tick_params(axis="x", rotation=30)
ax1.legend()

# Chart 2: is the speed stable, or are there spikes?
ax2.plot(total, lw=1)
ax2.axhline(total.mean(), ls="--", color="tab:orange", label=f"mean {total.mean():.1f} ms (~{fps_mean:.0f} FPS)")
ax2.axhline(p95, ls=":", color="tab:red", label=f"p95 {p95:.1f} ms (~{fps_p95:.0f} FPS)")
ax2.set_xlabel("frame")
ax2.set_ylabel("total ms per frame")
ax2.set_title("Total time per frame")
ax2.legend()

fig.tight_layout()
fig.savefig(RESULTS / "benchmark.png", dpi=150)
plt.show()

<div dir="rtl">

## كيف تحكيها بالمقابلة

- "قست سرعة كل مرحلة على ‏300‏ إطار من فيديو ثابت، بعد ‏10‏ إطارات تحمية، وبنفس الكود اللي بيشتغل لايف."
- "ركزت على ‏p95‏ مو بس المتوسط، لأنه باللايف القفزات بتبيّن كتقطيع."
- "الـ ‏FPS‏ هون للنماذج بس. باللايف الكاميرا نفسها محدودة، فالرقم الحقيقي بيطلع أقل أو مساوي."
- "أبطأ مرحلة هي [من الجدول]، فهي أول شي بحسّنه إذا احتجنا سرعة أكتر."

</div>